In [1]:
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator
import torch_geometric.transforms as T
from torch_geometric.utils import to_undirected
import torch
import numpy as np
import os.path as osp
from torch_geometric.nn import GCNConv, SAGEConv
import torch.nn.functional as F
from deeprobust.graph.utils import *
import torch_sparse
from deeprobust.graph.global_attack import PRBCD
import logging
import time
from utils import scale_feats

torch.cuda.set_device(2)  # 指第几块GPU
root_path = osp.expanduser('/home/songsh/GCL/datasets_pyg')

dataname = 'ogbn-arxiv'
dataset = PygNodePropPredDataset(root=osp.join(root_path, 'OGB'), name=dataname,
                                     transform=T.NormalizeFeatures())
data = dataset[0]
data.edge_index = to_undirected(data.edge_index, data.num_nodes)
device = torch.device('cuda')

# process features
feat = data.x 
data.x = scale_feats(feat)


# for arxiv import deeprobust.graph.utils as utils.add_mask
data = data.to(device)
split_idx = dataset.get_idx_split()
train_idx, val_idx, test_idx = split_idx["train"], split_idx["valid"], split_idx["test"]
n = data.x.shape[0]
data.train_mask = index_to_mask(train_idx, n)
data.val_mask = index_to_mask(val_idx, n)
data.test_mask = index_to_mask(test_idx, n)
data.y = data.y.squeeze()


ptb_rate = 0.5

/home/songsh/anaconda3/anaconda3_install/envs/ProGCL/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using backend: pytorch


In [2]:
agent = PRBCD(data, device=device)
start = time.time()
edge_index, edge_weight = agent.attack(ptb_rate=ptb_rate)
print(time.time() - start)
adj = torch.sparse_coo_tensor(indices=edge_index.cpu().numpy(), values=edge_weight.cpu().numpy(), size=[n, n])
adj = to_scipy(adj)
perturbed_adj = torch_sparse.tensor.from_scipy(adj).to(device)

GCN(
  (layers): ModuleList(
    (0): GCNConv(128, 256)
    (1): GCNConv(256, 256)
    (2): GCNConv(256, 40)
  )
  (bns): ModuleList(
    (0): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
)
=== training GCN model ===
Epoch 0, training loss: 4.431150913238525
Epoch 50, training loss: 0.9041182994842529
Epoch 100, training loss: 0.7990514636039734
Epoch 150, training loss: 0.7167672514915466
Epoch 200, training loss: 0.6317435503005981
Epoch 250, training loss: 0.5757774710655212
Epoch 300, training loss: 0.6233214735984802
Epoch 350, training loss: 0.5396972894668579
=== early stopping at 153, acc_val = 0.7248229806369342 ===
GCN Test set results: 0.7161697837582042
n_perturbations: 231559


  0%|          | 0/400 [00:00<?, ?it/s]

Epoch 0: -0.6398956775665283


  2%|▎         | 10/400 [00:20<13:45,  2.12s/it]

Epoch 10: 0.25023353099823


  5%|▌         | 20/400 [00:39<12:37,  1.99s/it]

... (73 outputs left out)

 98%|█████████▊| 390/400 [12:18<00:18,  1.83s/it]

Epoch 390: 0.7155523896217346


100%|██████████| 400/400 [12:36<00:00,  1.89s/it]


best_loss: 2.0370278358459473
1-th sampling: too many samples 231714.0
2-th sampling: too many samples 232032.0
3-th sampling: too many samples 232254.0
7-th sampling: too many samples 231822.0
8-th sampling: too many samples 232207.0
9-th sampling: too many samples 231708.0
16-th sampling: too many samples 232332.0
Test:
loss= 2.0029 accuracy= 0.3024
Validatoin:
loss= 2.0370 accuracy= 0.2919
762.7306933403015


In [3]:
def get_logger(filename, verbosity=1, name=None):
    level_dict = {0: logging.DEBUG, 1: logging.INFO, 2: logging.WARNING}
    formatter = logging.Formatter(
        "[%(asctime)s][%(filename)s][line:%(lineno)d][%(levelname)s] %(message)s"
    )
    logger = logging.getLogger(name)
    logger.setLevel(level_dict[verbosity])

    fh = logging.FileHandler(filename, "w")
    fh.setFormatter(formatter)
    logger.addHandler(fh)

    sh = logging.StreamHandler()
    sh.setFormatter(formatter)
    logger.addHandler(sh)

    return logger

logger = get_logger('./log/PRBCD_attack_%s_%s.log' % ('ogb', str(ptb_rate)))

In [4]:
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers,
                 dropout):
        super(GCN, self).__init__()

        self.convs = torch.nn.ModuleList()
        self.convs.append(GCNConv(in_channels, hidden_channels, cached=True))
        self.bns = torch.nn.ModuleList()
        self.bns.append(torch.nn.BatchNorm1d(hidden_channels))
        for _ in range(num_layers - 2):
            self.convs.append(
                GCNConv(hidden_channels, hidden_channels, cached=True))
            self.bns.append(torch.nn.BatchNorm1d(hidden_channels))
        self.convs.append(GCNConv(hidden_channels, out_channels, cached=True))

        self.dropout = dropout

    def reset_parameters(self):
        for conv in self.convs:
            conv.reset_parameters()
        for bn in self.bns:
            bn.reset_parameters()

    def forward(self, x, adj_t):
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x, adj_t)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.convs[-1](x, adj_t)
        return x.log_softmax(dim=-1)
    
    
def train(model, x, adj, labels, train_idx, optimizer):
    model.train()

    optimizer.zero_grad()
    out = model(x, adj)[train_idx]
    loss = F.nll_loss(out, labels.squeeze(1)[train_idx])
    loss.backward()
    optimizer.step()

    return loss.item()


@torch.no_grad()
def test(model, x, adj, labels, split_idx, evaluator):
    model.eval()

    out = model(x, adj)
    y_pred = out.argmax(dim=-1, keepdim=True)

    train_acc = evaluator.eval({
        'y_true': labels[split_idx['train']],
        'y_pred': y_pred[split_idx['train']],
    })['acc']
    valid_acc = evaluator.eval({
        'y_true': labels[split_idx['valid']],
        'y_pred': y_pred[split_idx['valid']],
    })['acc']
    test_acc = evaluator.eval({
        'y_true': labels[split_idx['test']],
        'y_pred': y_pred[split_idx['test']],
    })['acc']

    return train_acc, valid_acc, test_acc

In [5]:
from ogb.nodeproppred import Evaluator
evaluator = Evaluator(name='ogbn-arxiv')

gcn = GCN(in_channels=data.num_features,
          hidden_channels=256,
          out_channels=dataset.num_classes,
          num_layers=3,
          dropout=0.5).to(device)
for run in range(1):
    gcn.reset_parameters()
    optimizer = torch.optim.Adam(gcn.parameters(), lr=0.01)
    for epoch in range(1, 501):
        loss = train(gcn, data.x, perturbed_adj, data.y.unsqueeze(1),  train_idx, optimizer)
        result = test(gcn, data.x, perturbed_adj, data.y.unsqueeze(1),  split_idx, evaluator)
        if (epoch % 10) == 0:
            train_acc, valid_acc, test_acc = result
            logger.info(f'Run: {run + 1:02d}, '
                  f'Epoch: {epoch:02d}, '
                  f'Loss: {loss:.4f}, '
                  f'Train: {100 * train_acc:.2f}%, '
                  f'Valid: {100 * valid_acc:.2f}% '
                  f'Test: {100 * test_acc:.2f}%')

INFO:root:Run: 01, Epoch: 10, Loss: 1.9703, Train: 42.84%, Valid: 48.88% Test: 48.28%
[2023-12-10 15:00:04,474][2525669187.py][line:17][INFO] Run: 01, Epoch: 10, Loss: 1.9703, Train: 42.84%, Valid: 48.88% Test: 48.28%
INFO:root:Run: 01, Epoch: 20, Loss: 1.7717, Train: 46.43%, Valid: 53.26% Test: 53.62%
[2023-12-10 15:00:05,696][2525669187.py][line:17][INFO] Run: 01, Epoch: 20, Loss: 1.7717, Train: 46.43%, Valid: 53.26% Test: 53.62%
INFO:root:Run: 01, Epoch: 30, Loss: 1.6504, Train: 52.19%, Valid: 56.14% Test: 55.80%
[2023-12-10 15:00:06,913][2525669187.py][line:17][INFO] Run: 01, Epoch: 30, Loss: 1.6504, Train: 52.19%, Valid: 56.14% Test: 55.80%
INFO:root:Run: 01, Epoch: 40, Loss: 1.5692, Train: 55.44%, Valid: 58.01% Test: 57.48%
[2023-12-10 15:00:08,131][2525669187.py][line:17][INFO] Run: 01, Epoch: 40, Loss: 1.5692, Train: 55.44%, Valid: 58.01% Test: 57.48%
INFO:root:Run: 01, Epoch: 50, Loss: 1.5117, Train: 57.66%, Valid: 58.90% Test: 57.74%
[2023-12-10 15:00:09,351][2525669187.py][l

In [ ]:
#save
import scipy
from datasets_dgl.utils import sparse_mx_to_torch_sparse_tensor
save_path = '/home/songsh/SPMGAE/datasets_dgl/all_data_attack/PRBCD/'

perturbed_adj = sparse_mx_to_torch_sparse_tensor(adj)
features  = data.x.cpu()
labels    = data.y.cpu() # 需要squeeze到一维,这里之前处理过了
train_idx = train_idx.cpu()
val_idx   = val_idx.cpu()
test_idx  = test_idx.cpu()


torch.save(perturbed_adj,   save_path +  '%s/PRBCD_%s_%s.pt' % (dataname,  dataname, ptb_rate))

if ptb_rate == 0: #保存一次label和features即可
    scipy.sparse.save_npz(save_path + '%s/%s_features' % (dataname,  dataname), scipy.sparse.csr_matrix(np.array(features)))
    np.save(save_path + '%s/%s_labels' % (dataname, dataname),   labels)

np.save( save_path + '%s/PRBCD_%s_%s_idx_train' % (dataname, dataname, ptb_rate),   train_idx)
np.save( save_path + '%s/PRBCD_%s_%s_idx_val'   % (dataname, dataname, ptb_rate),   val_idx  )
np.save( save_path + '%s/PRBCD_%s_%s_idx_test'  % (dataname, dataname, ptb_rate),   test_idx )

In [6]:
from   datasets_dgl.data_dgl  import load_attack_data
from     easydict        import EasyDict

# dataset_name = dataset_name.split('-')[1]
DATASET = EasyDict()
DATASET.ATTACK = {
    "data"    : 'Attack-ogbn-arxiv',
    "attack"  : 'PRBCD',
    "ptb_rate":  str(ptb_rate)
}
# now just attack use
dataset  = load_attack_data(DATASET['ATTACK'])
graph = dataset.graph
graph

[I] Loading dataset Attack-ogbn-arxiv...
train_mask, test_mask, val_mask sizes : tensor(90941) tensor(29799) tensor(48603)
[I] Finished loading.
[I] Data load time: 4.3432s


Graph(num_nodes=169343, num_edges=2947503,
      ndata_schemes={'feat': Scheme(shape=(128,), dtype=torch.float32), 'label': Scheme(shape=(), dtype=torch.int64), 'train_mask': Scheme(shape=(), dtype=torch.bool), 'val_mask': Scheme(shape=(), dtype=torch.bool), 'test_mask': Scheme(shape=(), dtype=torch.bool)}
      edata_schemes={})